In [ ]:
#http://localhost:9200/hawker_reviews/_search?pretty
from pprint import pprint
from elasticsearch import Elasticsearch,helpers
es = Elasticsearch(['http://localhost:9200'], basic_auth=("elastic", "5vRl7G_wpBvo8CytUL=h"))
client_info = es.info()
print("Connected to ElasticSearch")
pprint(client_info.body)

In [1]:
import json 
with open('reviews.json','r',encoding='utf-8') as f:
    reviews = json.load(f)
    unique_hawkers = set()
for review in reviews:
    hawker_name =  review.get('title','')
    if hawker_name:
        unique_hawkers.add(hawker_name)

print(f"Found {len(unique_hawkers)} unique hawkers,{unique_hawkers}")


Found 65 unique hawkers,{'Yishun Park Hawker Centre', 'Sembawang Hills Food Centre', 'Pek Kio Market & Food Centre', 'Punggol Coast Hawker Centre', 'Adam Food Centre', 'Pasir Panjang Food Centre', 'Chinatown Complex', 'Blk 724 Ang Mo Kio Ave 6', 'Hawker Centre @ Our Tampines Hub', 'Yuhua Village Market and Food Centre', 'East Coast Lagoon Food Village', 'Holland Drive Market & Food Centre', 'Tanglin Halt Market', 'Bedok 85 Market', 'Berseh Food Centre', 'Toa Payoh Lorong 8 Market & Hawker Centre', 'ABC Brickworks Market & Food Centre', 'Albert Centre Market & Food Centre', 'Taman Jurong Market & Food Centre', 'Old Airport Road Food Centre', 'Alexandra Village Food Centre', '20 Ghim Moh Road Market & Food Centre #01-06', 'Bukit Merah View Market & Hawker Centre', 'Promenade Market @ 84', 'Bendemeer Market & Food Centre', "People's Park Food Centre", 'Kaki Bukit 511 Market & Food Centre', 'Margaret Drive Hawker Centre', 'Lau Pa Sat', 'Dunman Food Centre', 'Bukit Merah Central Food Centre

In [ ]:
# Init unique ids to each hawker
import re
import hashlib 
from pprint import pprint
def hawker_id_from_name(name):
    name = name.lower()
    name = re.sub(r"[^a-z0-9]+", " ", name)
    name = name.strip()
    return hashlib.md5(name.encode()).hexdigest()

hawker_id_dict = {}

for hawker in unique_hawkers:
    hawker_id_dict[hawker]  = hawker_id_from_name(hawker)
pprint(hawker_id_dict)

    

{'20 Ghim Moh Road Market & Food Centre #01-06': 'aaada0af4eece0aebd83dde20e1ce5f1',
 '505 Jurong West Market & Food Centre': 'ab8d6aaff6be8d3edc7f12cd59f934d3',
 'ABC Brickworks Market & Food Centre': 'abf5251c210c665daafcbfc535ec7883',
 'Adam Food Centre': 'e5a08f310e41bc00f351c1be24f1d4bd',
 'Albert Centre Market & Food Centre': '23ca55c4562bb8f5638e611658f2a16d',
 'Alexandra Village Food Centre': '0ef327ef02a28ad20049ba3d81e1fd90',
 'Amoy Street Food Centre': 'cc243a333b87eb5d9e5a462319f2b243',
 'Ang Mo Kio 628 Market': '4547f4adda032f6f8116e7d38489fdec',
 'Bedok 538 Market & Food Centre': '01cb0889eda76eba43975dec82df2504',
 'Bedok 85 Market': '8f1ddeb7305ed5c710c3c3d7ef7c0b66',
 'Bedok Food Centre': '041fa07d7360036cc17802ffb6068b84',
 'Bendemeer Market & Food Centre': '059fd9f83ced7623ea14710817341ec4',
 'Beo Crescent Market': '58aef4f088d0e46dba113f8d70f14ad5',
 'Berseh Food Centre': '11a9bf7863fcf392b4d85e9b1f3d3ea2',
 'Blk 724 Ang Mo Kio Ave 6': '8973918edb5bd7d76f055fafcc7b5

In [ ]:
# add hawker id to new file and then index
with open('reviews_with_coords.json','r',encoding='utf-8') as f:
    reviews = json.load(f)
for review in reviews:
   hawker_name = review.get("title","")
   if hawker_name in hawker_id_dict:
        review["hawker_id"] = hawker_id_dict[hawker_name]

with open("reviews_with_coords_locId.json", "w",encoding="utf-8") as f:
  json.dump(reviews, f,indent=2, ensure_ascii =False)
print("Add hawker id to reviews")

Add location to reviews


In [ ]:
import  os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import requests
url ="https://www.onemap.gov.sg/api/auth/post/getToken"
payload = {
        "email":os.environ["ONEMAP_EMAIL"],
        "password": os.environ["ONEMAP_PASSWORD"]
    }

response=requests.request("POST",url,json=payload)

print(response.text)

In [ ]:
from urllib.parse import quote

def geocode_loc(query):
    encoded_query = quote(query,safe="&")
    url = f"https://www.onemap.gov.sg/api/common/elastic/search?searchVal={encoded_query}&returnGeom=Y&getAddrDetails=Y"
    # params = {"searchVal": query, "returnGeom": "Y","getAddrDetails":"Y"}
    headers = {"Authorization":os.environ["ONEMAP_TOKEN"]}
    response = requests.get(url,headers=headers)
    data = response.json()
    #print(f"Found: {data.get('found', 0)}")
   

    if data.get('found', 0) > 0:
        result = data['results'][0]
        return float(result['LATITUDE']), float(result['LONGITUDE'])

    print(data.get("error"))
    return None, None



geocode_loc("Geylang Serai Market and Food Centre ")

In [ ]:
hawker_dict = {}

manual_coords = {
    "Geylang East Market & Food Centre" : {"lat": 1.320681,"lon":103.886664},
    "Geylang Serai Market and Food Centre" : {"lat":1.316846 ,"lon":103.898282},
    "Whampoa Food Block 91 Food Centre" : {"lat": 1.323477,"lon":103.854081},
    "Bedok 85 Market" : {"lat":1.332117 ,"lon":103.9387361964381},
    "Hougang Avenue 1 Block 105 Market and Food Centre" : {"lat":1.354257 ,"lon":103.890022},
    "Promenade Market @ 84" : {"lat": 1.302430,"lon":103.906214},
}

for i, hawker_name in enumerate(unique_hawkers, 1):
    lat, lon = geocode_loc(hawker_name)
    if lat and lon:
        hawker_dict[hawker_name] ={"lat":lat, "lon":lon}
        print(f"{i}/{len(unique_hawkers)}: {hawker_name} ({lat:.4f}, {lon:.4f})")
    else:
        print(f"Failed: '{hawker_name}'")
        print(f"In manual_coords? {hawker_name in manual_coords}")

        if hawker_name in manual_coords:
            hawker_dict[hawker_name] = manual_coords[hawker_name]
        else:
            print(f"{hawker_name} lat and lon not found ")




In [ ]:
len(hawker_dict)

In [ ]:
for review in reviews:
    hawker_name = review.get("title","")
    if hawker_name in hawker_dict:
        review["location"] = hawker_dict[hawker_name]

with open("reviews_with_coords.json", "w",encoding="utf-8") as f:
  json.dump(reviews, f,indent=2, ensure_ascii =False)
print("Add location to reviews")

In [ ]:
#GET UNIQUE REVIEW CONTEXT KEYS
import json
from collections import Counter

with open("reviews_with_coords.json","r",encoding = "utf-8") as f:
 reviews = json.load(f)
all_fields = Counter()
for reviewObj in reviews:
    context =  reviewObj.get("reviewContext",{})
    all_fields.update(context.keys())

for field, count in all_fields.most_common():
    print(f"{field}: {count}")


In [ ]:
#GET UNIQUE REVIEW CONTEXT NUMERIC FIELD VALUES
import json
def uniqueVals(keyname):
  with open("reviews_with_coords.json","r",encoding = "utf-8") as f:
      reviews = json.load(f)
      all_vals  = set()
      for reviewObj in reviews:
       context =  reviewObj.get("reviewContext",{})
       numField = context.get(keyname)
       all_vals.add(numField)
      

      print(all_vals)

#uniqueVals("Price per person")
#uniqueVals("Wait time")
#uniqueVals("Group size")
for field in all_fields:
  print (field)
  uniqueVals(field)

In [ ]:
import re
def parse_price(text):
    if not text :
        return None,None
    nums = list(map(int, re.findall(r"\d+", text)))
    if not nums:
        return None, None

    elif "+" in text:
        return nums[0], None
    
    else: 
        return nums[0], nums[1]
    
print(parse_price("$10–20"))
print(parse_price("$100+"))
print(parse_price(None))

In [ ]:
def parse_wait_time(text):
     if not text:
        return None,None
     if "No wait" in text:
         return 0,0
     nums = list(map(int, re.findall(r"\d+", text)))
     if "hour" in text:
        mins = nums[0] * 60
        if "+" in text:
         return mins, None
        return mins,mins
     if "Up to" in text:
        return 0, nums[0]
     if len(nums) == 1:
        return nums[0], nums[0]
     if len(nums) >= 2:
        return nums[0], nums[1]
     return None,None

print(parse_wait_time("10-30 min"))
print(parse_wait_time("No wait"))
print(parse_wait_time("Up to 10 min"))
print(parse_wait_time("1+ hour"))
print(parse_wait_time(None))

In [ ]:
mapping = {
    "settings": {
        "analysis": {
            "analyzer": {
                "folding_analyzer": {
                    "tokenizer": "standard",
                    "filter": ["lowercase", "asciifolding"]
                }
            },
            "normalizer": {
                "lowercase_normalizer": {
                    "type": "custom",
                    "filter": ["lowercase","asciifolding"]
                }
            }
        }
    },
    "mappings":{
        "properties":{
            "hawker_name":{
                "type": "text",
                "analyzer": "folding_analyzer",
                "fields": {
                    "keyword": {"type": "keyword",
                                "normalizer":"lowercase_normalizer"
                            }
                        }
            },
            "hawker_id": {
            "type": "keyword"
            },
            "location": {
                "type": "geo_point"
            },
            "review_text": {
                "type": "text",
                "analyzer": "english"
            },
            "review_rating": {"type": "integer"},
            "review_author": {"type": "keyword"},
            "review_date": {"type": "date"},

            "food_rating": {"type": "integer"},
            "service_rating": {"type": "integer"},
            "atmosphere_rating": {"type": "integer"},
             

            "review_context": {"type": "flattened"},

            "context_wait_min": { "type": "integer" },
            "context_wait_max": { "type": "integer" },
            "context_price_min": { "type": "integer" },
            "context_price_max": { "type": "integer" }, 
            "context_parking_space": { "type": "keyword" },

            "context_recommended": {
               "type": "text",
               "analyzer": "folding_analyzer",
               "fields": {
                "raw": { "type": "keyword",
                        "normalizer": "lowercase_normalizer"
                    }
                }
            },

              "context_meal_type": { "type": "keyword" },    
         }
    }
}

index_name = "hawker_reviews"
if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)

es.indices.create(index=index_name, body=mapping)




In [ ]:
import json
with open("reviews_with_coords_locId.json","r",encoding = "utf-8") as f:
    reviews = json.load(f)
count = 0 
actions = []
for reviewObj in reviews :
    context = reviewObj.get('reviewContext', {})
    location = reviewObj.get('location') 
    detailed_rating = reviewObj.get('reviewDetailedRating', {})
    doc = {
            "hawker_name": reviewObj.get("title",""),
            "hawker_id": reviewObj.get("hawker_id",""),
            "location": location,
            "review_text": reviewObj.get('text', ''),
            #"review_rating": reviewObj.get('stars', 0),
            "review_author":reviewObj.get('name', 'Anonymous'),
            "review_date": reviewObj.get('publishedAtDate', ''),
            
            
          
           
          
            "review_context": context if context else {}

            

    }

    if reviewObj.get("stars"):
       doc["review_rating"] = reviewObj.get("stars")
    if detailed_rating.get('Food'):
            doc["food_rating"] = detailed_rating['Food']
    if detailed_rating.get('Service'):
            doc["service_rating"] = detailed_rating['Service']
    if detailed_rating.get('Atmosphere'):
            doc["atmosphere_rating"] = detailed_rating['Atmosphere']

    
    #review context extraction 
    if context.get("Wait time"):
        doc["context_wait_min"], doc["context_wait_max"] = parse_wait_time(context.get("Wait time"))
    if context.get("Price per person"):
        doc["context_price_min"],doc["context_price_max"] = parse_price(context.get("Price per person"))

    if context.get("Parking space"):
         doc["context_parking_space"] =  context.get("Parking space")
    if context.get("Recommended dishes"):
         doc["context_recommended"] =  context.get("Recommended dishes")
    if context.get("Meal type"):
         doc["context_meal_type"] = context.get("Meal type")
   
    if doc['review_text']:
        actions.append({
            "_index": index_name,
            "_source": doc
        })
        count += 1
    
helpers.bulk(es, actions)
print(f"Indexed {count} reviews")      

In [ ]:
#index_mapping = es.indices.get_mapping(index=index_name)
#pprint(index_mapping["hawker_reviews"]["mappings"]["properties"])

In [ ]:
sg_hawker_dishes = [
    # rice dishes
    "hainanese chicken rice",
    "roast chicken rice",
    "roast pork rice",
    "char siew rice",
    "duck rice",
    "braised duck rice",
    "claypot rice",
    "nasi lemak",
    "nasi padang",
    "nasi briyani",
    "nasi goreng",
    "beef rendang rice",
    "ayam penyet",
    "ayam penyet rice",
    "salted egg chicken rice",
    "curry chicken rice",
    "economy rice",
    "mixed rice",
    "teochew porridge",
    "pork chop rice",
    "sweet and sour pork rice",
    "porridge"

    # noodles (dry & soup)
    "char kway teow",
    "hokkien mee",
    "fried hokkien prawn mee",
    "prawn mee",
    "prawn noodle soup",
    "bak chor mee",
    "bah chok mee",
    "bcm",
    "minced pork noodles",
    "lor mee",
    "wanton mee",
    "wanton noodle soup",
    "fishball noodles",
    "fishball noodle soup",
    "laksa",
    "katong laksa",
    "mee siam",
    "mee rebus",
    "mee goreng",
    "maggi goreng",
    "indomie goreng",
    "dry ban mian",
    "soup ban mian",
    "you mian",
    "ee mian",
    "duck noodles",
    "fish head bee hoon",
    "fish soup",
    "sliced fish soup",
    "seafood noodles",
    "beef noodles",
    "kway chap",
    "pig organ soup",
    "pig organ noodles",

    # indian & muslim food
    "roti prata",
    "egg prata",
    "cheese prata",
    "plain prata",
    "prata with curry",
    "murtabak",
    "thosai",
    "masala thosai",
    "idli",
    "mee soto",
    "soto ayam",
    "sup kambing",
    "mutton soup",
    "beef rendang",
    "chicken rendang",
    "ayam merah",
    "briyani chicken",
    "briyani mutton",
    "mee kuah",
    "mee goreng ayam",

    # chinese classics
    "chwee kueh",
    "chee cheong fun",
    "yong tau foo",
    "popiah",
    "oyster omelette",
    "orh luak",
    "orh lua"
    "carrot cake",
    "fried carrot cake",
    "white carrot cake",
    "black carrot cake",
    "fried oyster omelette",
    "satay bee hoon",
    "lor bak",
    "ngoh hiang",
    "fried spring rolls",
    "fried dumplings",
    "shui jiao",
    "dumplings"
    "xiao long bao",
    "steam fish",
    "steamed fish"

    # bbq / grilled
    "satay",
    "chicken satay",
    "mutton satay",
    "pork satay",
    "grilled stingray",
    "bbq stingray",
    "bbq squid",
    "bbq sambal seafood",
    "grilled chicken wings",
    "bbq chicken wings",
    "otah",
    "otah otah",
    "roast meat"

    # soup & herbal
    "bak kut teh",
    "pepper bak kut teh",
    "herbal bak kut teh",
    "chicken herbal soup",
    "lotus root soup",
    "old cucumber soup",
    "abc soup",
    "fish maw soup",
    "winter melon soup",
    "fish soup"

    # snacks & sides
    "fried chicken wings",
    "fried chicken cutlet",
    "fried tofu",
    "fried fish cake",
    "prawn fritters",
    "you tiao",
    "chinese dough fritters",
    "fried mantou",
    "curry puff",
    "samosa",
    "spring rolls",
    "popcorn chicken",
    "bbq wings",
    "eggs"

    # desserts & sweets
    "ice kacang",
    "chendol",
    "tau huay",
    "soy bean curd",
    "grass jelly",
    "cheng tng",
    "red bean soup",
    "green bean soup",
    "black sesame paste",
    "peanut soup",
    "bubur cha cha",
    "sweet potato soup",
    "ondeh ondeh",
    "kueh lapis",
    "ang ku kueh",

    # drinks
    "teh tarik",
    "teh peng",
    "teh o",
    "teh o peng",
    "kopi",
    "kopi o",
    "kopi c",
    "kopi peng",
    "milo",
    "milo peng",
    "bandung",
    "barley water",
    "chrysanthemum tea",
    "sugarcane juice",
    "lime juice"
]


In [ ]:
"""
def parse_dish_query(query,dishes):
    q = query.lower()
    for dish in dishes:
        if dish in q: 
          return dish
    return None


def parse_time_query(query):
   q = query.lower()
   patterns = [
       r"wait\s*[<≤]?\s*(\d+)\s*min",
       r'under\s*(\d+)\s*min',
       r'less than\s*(\d+)\s*min'

    ]
   for p in patterns: 
      match = re.search(p,q)
      if match:
         return  int(match.group(1))

   return None;


PRICE_KEYWORDS = {
    "cheap",
    "very cheap",
    "affordable"
}
def parse_price_query(query):
   q = query.lower()
   match = re.search(r'under\s*\$?(\d+)', q)
   if match:
     return int(match.group(1))

   for cheap in PRICE_KEYWORDS():
        if cheap in q :
            return True;
   return None
    


   # parse meal type
"""

In [ ]:
SLANG_MAP = {
    "shiok": "delicious",
    "tahan": "tolerate",
    "meh": "average"
}


def normalize_text(text):
    text= text.lower()
    for slang,replacement in SLANG_MAP.items():
      text = text.replace(slang,replacement)
    return text

def split_sentence(text):
    sentences =  re.split(r"[.!?]",text)
    return [s.strip() for s in sentences if s.strip()]

def extract_dishes(text, dishes=sg_hawker_dishes):
   return [d for d in dishes if d in text]

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()
def get_sentiment_score(text):
   return analyzer.polarity_scores


def process_reviews_basic(reviews):
   for reviews 

In [ ]:
from fastapi import FastAPI
app = FastAPI()
@app.get("/")
def root():
    return {"Hello":"World"}


